# Task 1 — Inconsistencies and duplicates

## Setup

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)

## Load the data

In [2]:
raw = pd.read_csv("club_signups.csv")
print("Shape:", raw.shape)
raw.head()

Shape: (39, 9)


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Soccer,2026-09-15 13:39,N
1,S010,61-3846,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,Management,alexandria,Chess,15/09/2026 03:12,false
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32,Y
3,S035,61-4844,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,Engineering and Materials Science,Cairo,Debating,2026-09-17 11:15,yes
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess Club,2026-09-13 22:36,yes


## Part A — Inconsistencies
### 1. First look
Show the shape, dtypes, and value_counts() of every categorical column, so the problems are visible before fixing them.

In [3]:
raw.dtypes

signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object

In [4]:
for col in ["faculty", "club", "city", "fee_paid"]:
    print(f"\n--- {col} ---")
    print(raw[col].value_counts(dropna=False))


--- faculty ---
faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64

--- club ---
club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music          2
chess          2
debate club    1
Music          1
Debate         1
Name: count, dtype: int64

--- city ---
city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza   

Make a working copy. From here on, `raw` is never touched.

In [5]:
signups = raw.copy()

### 2. Faculty, club, city
Map every variant to its canonical value. Normalise mechanically first (strip spaces, lower-case), then use an explicit dictionary for what's left. Assert that only canonical values remain.

In [6]:
# Faculty
FACULTIES = {"MET", "IET", "EMS", "Pharmacy", "Management"}
FACULTY_ALIASES = {
    "media engineering and technology": "MET",
    "information engineering and technology": "IET",
    "engineering and materials science": "EMS",
    "pharma": "Pharmacy"
}

faculty = signups["faculty"].str.strip().str.lower()
signups["faculty"] = faculty.map(FACULTY_ALIASES).fillna(faculty.str.title())

# Fix casing for abbreviations that title() would turn into Met, Iet, Ems
signups["faculty"] = signups["faculty"].replace({"Met": "MET", "Iet": "IET", "Ems": "EMS"})

assert set(signups["faculty"]) == FACULTIES, set(signups["faculty"]) - FACULTIES
signups["faculty"].value_counts()

faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64

In [7]:
# Club
CLUBS = {"Robotics", "Debate", "Music", "Football", "Chess"}
CLUB_ALIASES = {
    "soccer": "Football",
    "chess club": "Chess",
    "debate club": "Debate",
    "debating": "Debate"
}

club = signups["club"].str.strip().str.lower()
signups["club"] = club.map(CLUB_ALIASES).fillna(club.str.title())

assert set(signups["club"]) == CLUBS, set(signups["club"]) - CLUBS
signups["club"].value_counts()

club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64

In [8]:
# City
CITIES = {"Cairo", "Giza", "Alexandria"}
CITY_ALIASES = {
    "alex": "Alexandria",
    "el giza": "Giza"
}

city = signups["city"].str.strip().str.lower()
signups["city"] = city.map(CITY_ALIASES).fillna(city.str.title())

assert set(signups["city"]) == CITIES, set(signups["city"]) - CITIES
signups["city"].value_counts()

city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64

### 3. Names and emails
One spelling per person: no extra spaces anywhere in names, and the same capitalisation for everyone. Emails trimmed and lower-case.

In [9]:
signups["full_name"] = signups["full_name"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
signups["email"] = signups["email"].str.strip().str.lower()

signups[["full_name", "email"]].head()

,full_name,email
0,Hana Zaki,hana.zaki@student.guc.edu.eg
1,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg
2,Mohamed Adel,mohamed.adel@student.guc.edu.eg
3,Mohamed Adel,mohamed.adel@student.guc.edu.eg
4,Laila Mostafa,laila.mostafa@student.guc.edu.eg


### 4. fee_paid
Map to a boolean column (True/False).

In [10]:
# Map strings to booleans
fee_paid = signups["fee_paid"].astype(str).str.strip().str.lower()
FEE_TRUE = {"yes", "y", "true", "1"}
FEE_FALSE = {"no", "n", "false", "0"}

signups["fee_paid"] = fee_paid.map(lambda x: True if x in FEE_TRUE else False if x in FEE_FALSE else pd.NA).astype("boolean")

assert pd.api.types.is_bool_dtype(signups["fee_paid"])
signups["fee_paid"].value_counts()

fee_paid
True     27
False    12
Name: count, dtype: Int64

### 5. signed_up_at
Two formats are mixed: `YYYY-MM-DD HH:MM` and `DD/MM/YYYY HH:MM`. We find them, check the parts, and parse.

In [11]:
# Find rows with a slash
is_slash = signups["signed_up_at"].str.contains("/", regex=False)

# Check the first part of the slash dates to see if it goes above 12
first_part = signups.loc[is_slash, "signed_up_at"].str[:2].astype(int)
print("first part > 12: ", (first_part > 12).sum())

first part > 12:  14


Because the first part of the slash dates goes above 12, it must be the day (`DD/MM`).

In [12]:
parsed = pd.Series(pd.NaT, index=signups.index, dtype="datetime64[ns]")

# ISO format rows (no slash)
parsed[~is_slash] = pd.to_datetime(signups.loc[~is_slash, "signed_up_at"], format="%Y-%m-%d %H:%M")
# Slash format rows (day first)
parsed[is_slash] = pd.to_datetime(signups.loc[is_slash, "signed_up_at"], format="%d/%m/%Y %H:%M")

signups["signed_up_at"] = parsed
assert signups["signed_up_at"].notna().all(), "Some timestamps failed to parse"
signups["signed_up_at"].head()

0   2026-09-15 13:39:00
1   2026-09-15 03:12:00
2   2026-09-14 23:32:00
3   2026-09-17 11:15:00
4   2026-09-13 22:36:00
Name: signed_up_at, dtype: datetime64[ns]

## Part B — Duplicates
### 1. Exact duplicates
Count them, then remove the extra copies.

In [13]:
exact_dups = signups.duplicated().sum()
print(f"Exact duplicate rows: {exact_dups}")
signups[signups.duplicated(keep=False)].sort_values("signup_id")

Exact duplicate rows: 3


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
34,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
15,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
24,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
20,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
33,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False


In [14]:
before_exact = len(signups)
signups = signups.drop_duplicates().reset_index(drop=True)
after_exact = len(signups)
print(f"Removed {before_exact - after_exact} exact duplicates. Rows left: {after_exact}")

Removed 3 exact duplicates. Rows left: 36


### 2. The same sign-up submitted again
Keep one row per `student_id` and `club`: the latest submission. Count how many rows this removes.

In [15]:
same_signup = signups.duplicated(subset=["student_id", "club"]).sum()
print(f"Rows with repeated student_id and club: {same_signup}")
signups[signups.duplicated(subset=["student_id", "club"], keep=False)].sort_values(["student_id", "club"])

Rows with repeated student_id and club: 4


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-16 09:07:00,True
3,S035,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-17 11:15:00,True
22,S021,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-16 01:15:00,False
24,S034,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-16 02:01:00,True
32,S017,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-15 18:01:00,False
21,S032,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-17 23:36:00,False
35,S036,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-18 23:36:00,True


In [16]:
before_dedup = len(signups)
signups = signups.sort_values("signed_up_at").drop_duplicates(subset=["student_id", "club"], keep="last").reset_index(drop=True)
after_dedup = len(signups)
print(f"Removed {before_dedup - after_dedup} older submissions. Rows left: {after_dedup}")

Removed 4 older submissions. Rows left: 32


### 3. Check unique combination
Assert that the combination `student_id` + `club` is unique.

In [17]:
assert signups.duplicated(subset=["student_id", "club"]).sum() == 0, "student_id + club is not unique"
print("✓ student_id + club is unique")

✓ student_id + club is unique


## Save the result
Save as `club_signups_clean.csv`.

In [18]:
signups.to_csv("club_signups_clean.csv", index=False)
print(f"Saved {len(signups)} clean rows to club_signups_clean.csv")
signups.head()

Saved 32 clean rows to club_signups_clean.csv


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
2,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
3,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
4,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
